# 10 - Full Dual-Track Pipeline (Path 3, A->M)

**Inputs needed:** Raw DICOMs, `data/labels.csv`, `configs/default.yaml`.
**Outputs produced:** Every artefact described in `ARCHITECTURE.md` (Phase 2 -> 6) plus optional TCAV outputs.
**Runtime:** End-to-end demo budget: ~2 hours on a single GPU for a small cohort.


End-to-end execution of the design doc, combining both paths:

```
A1/A2 -> B (phase filter + DICOM->NIfTI)
      -> C (HU + Z-score)
      -> D (segmentation)
      -> E (bbox crop)
      -> F (PyRadiomics)              -> H (VaRFS)               -> I (stable radiomics vector)
      -> G (3D SwinViT input)         -> J (self-attention)      -> K (deep visual vector)
                                                                  \
                                                                   -> L (cross-attention fusion)
                                                                                       \
                                                                                        -> M (HCC probability)
```

End artifacts:
- All Phase 2 + 3 + 4 outputs from notebooks 08 and 09.
- `models/saved/fused_cross_attention.pth` and `results/training_history_fusion.json`.
- `results/evaluation_metrics.json` for radiomics / SwinViT / fused.
- `Plots/roc_comparison.png`, `Plots/ablation_bar_chart.png`, `Plots/training_curves.png`.
- `attention/<PID>_attention.nii.gz` per validation patient.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch
from src.utils.config import ensure_dirs, load_config, load_features_config, set_seed
from src.utils.logger import setup_logger

cfg = load_config(ROOT / "configs" / "default.yaml")
features_cfg = load_features_config(ROOT / "configs" / "features.yaml")
ensure_dirs(cfg)
set_seed(int(cfg.get("seed", 42)))
setup_logger("hcc", log_file=Path(cfg["paths"]["logs_dir"]) / "full_pipeline.log")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Stage 1 - Phase 2 preprocessing (B->E)

In [ ]:
from tqdm.notebook import tqdm
from src.data.dicom_loader import DICOMLoader
from src.data.liver_segmentation import LiverSegmentor
from src.data.preprocessing import preprocess_volume
from src.data.cropping import crop_patient

raw_dir = Path(cfg["paths"]["raw_dir"])
processed_dir = Path(cfg["paths"]["processed_dir"])
loader = DICOMLoader(raw_dir, cfg["preprocessing"]["target_phase"], processed_dir)
segmentor = LiverSegmentor(
    gpu=bool(cfg["preprocessing"]["liver_segmentation"].get("gpu", True))
)
patient_ids = sorted(p.name for p in raw_dir.iterdir() if p.is_dir())
for pid in tqdm(patient_ids, desc="Phase 2"):
    out_dir = processed_dir / pid
    if (out_dir / "before_cropped.nii.gz").exists() and (out_dir / "crop_metadata.json").exists():
        continue
    try:
        volume_path = loader.convert_to_nifti(pid)
        mask_path = segmentor.segment(volume_path)
        _, stats = preprocess_volume(volume_path, mask_path, cfg)
        _, _, _ = crop_patient(volume_path.parent, zscore_stats=stats)
    except Exception as exc:
        print(f"Skip {pid}: {exc}")


## Stage 2 - Radiomics path (F->H->I)

In [ ]:
from src.features.radiomics_extractor import RadiomicsExtractor
from src.features.varfs_selection import VaRFSSelector
from src.features.baseline_classifier import RadiomicsBaseline, save_metrics

raw_csv = ROOT / "data" / "raw_radiomics_features.csv"
raw_features = RadiomicsExtractor(features_cfg["radiomics"]).extract_all(
    processed_dir=processed_dir,
    labels_csv=cfg["paths"]["labels_csv"],
    output_csv=raw_csv,
)
varfs_cfg = features_cfg["varfs"]
selector = VaRFSSelector(
    n_bootstrap=int(varfs_cfg["n_bootstrap"]),
    stability_threshold=float(varfs_cfg["stability_threshold"]),
    correlation_threshold=float(varfs_cfg["correlation_threshold"]),
    top_k_per_iter=int(varfs_cfg["top_k_per_iter"]),
    use_robust_scaler=bool(varfs_cfg["use_robust_scaler"]),
    random_state=int(cfg["seed"]),
)
selector.fit(raw_features, raw_features["label"])
filtered = selector.transform(raw_features)
filtered.to_csv(ROOT / "data" / "varfs_filtered_features.csv", index=False)
selector.save_selection(ROOT / "data" / "varfs_selected_features.json")

baseline = RadiomicsBaseline(features_cfg["baseline"])
baseline_metrics = baseline.train(filtered)
baseline.save(Path(cfg["paths"]["model_save_dir"]) / "radiomics_baseline.pkl")
save_metrics(baseline_metrics, Path(cfg["paths"]["results_dir"]) / "baseline_metrics.json")
baseline_metrics

## Stage 3 - SwinViT path (G->J->K)

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("run_training", ROOT / "scripts" / "run_training.py")
run_training = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run_training)
run_training._train_swinvit(cfg)

## Stage 4 - Cross-attention fusion (L) + classifier head (M)

Loads stable radiomics + SwinViT deep vectors and trains the joint embedding.

In [ ]:
run_training._train_fusion(cfg)

## Stage 5 - Phase 6 evaluation

Runs the same logic as `scripts/run_evaluation.py`: produces metrics for
radiomics / SwinViT / fused and saves ROC, ablation, and training-curve plots.

In [ ]:
spec = importlib.util.spec_from_file_location("run_evaluation", ROOT / "scripts" / "run_evaluation.py")
run_evaluation = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run_evaluation)
import sys as _sys
_sys.argv = ["run_evaluation.py", "--config", str(ROOT / "configs" / "default.yaml")]
run_evaluation.main()

## Stage 6 - Compare the three approaches (ablation)

In [ ]:
import json
import pandas as pd
from IPython.display import Image

metrics_path = Path(cfg["paths"]["results_dir"]) / "evaluation_metrics.json"
metrics = json.loads(metrics_path.read_text())
df = pd.DataFrame(metrics).T
df

In [ ]:
plots_dir = Path(cfg["paths"]["plots_dir"])
for name in ("roc_comparison.png", "ablation_bar_chart.png", "training_curves.png"):
    fp = plots_dir / name
    if fp.exists():
        display(Image(filename=str(fp)))

## Stage 7 - Inference on a new patient (M)

End-to-end inference using the trained dual-track model.

In [ ]:
DICOM_DIR = "/path/to/new_patient/before/"  # <-- edit me
PATIENT_ID = "INF_full_pipeline"
!python scripts/run_inference.py \
    --config configs/default.yaml \
    --dicom_dir "$DICOM_DIR" \
    --patient_id $PATIENT_ID \
    --save_heatmap